# `transformer_model` -- Causal Transformer (Secondary/Alternative Model)

**Ported near-verbatim from `project_hrishav/sequence_model.py`.** This is hrishav's core architectural contribution: instead of hand-crafted aggregate features at one point in the innings (project_gagan's approach), model the *entire sequence* of deliveries with a Transformer, predicting win probability (and, optionally, next-ball outcome and final score) at every single ball.

**Why this file needed almost no changes to port**: the architecture only operates on numeric feature/embedding tensors -- it has no idea whether those numbers came from Cricsheet JSON or `ipl_data.xlsx`. The only thing that changed is where the hyperparameter constants come from: hrishav's original code imported them from a `config.py` module; this port inlines the same numeric values directly at the top of this file.

**This is explicitly the SECONDARY model, not the default.** hrishav's own bootstrap-CI statistical testing found this architecture does *not* beat a simple calibrated Logistic Regression at a statistically significant level, and that the auxiliary next-ball/score prediction heads actively *hurt* the primary win-probability signal (hrishav's own "Task 4" finding). This port's default hyperparameters (`LAMBDA_NEXT_BALL=0`, `LAMBDA_SCORE=0`) already bake in that lesson -- by default this trains "win-only", matching hrishav's own recommended configuration -- but the full multi-task architecture and loss function are kept completely intact, so multi-task training is still available to anyone who passes nonzero lambdas.

In [1]:
from typing import Optional

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

## Hyperparameter constants

All values match `project_hrishav/config.py` exactly:
- `GAME_STATE_DIM = 24`: the hand-crafted per-ball vector from   `src/game_state.py`.
- `PLAYER_EMBED_DIM = 32`: dimensionality of each player's   learned/random embedding.
- `INPUT_DIM = 24 + 3*32 = 120`: the game-state vector   concatenated with three player embeddings (batter, bowler,   non-striker) -- this is what actually goes into the   Transformer at every ball.
- `TRANSFORMER_D_MODEL/NHEAD/NUM_LAYERS/DIM_FF/DROPOUT`: a   fairly small Transformer by modern standards (2 layers, 4   heads, 64-dim) -- appropriately sized for a dataset with only   a few thousand innings, where a larger model would almost   certainly overfit.
- `MAX_SEQ_LEN = 280`: the longest an innings' delivery sequence   is ever padded/truncated to.
- `NEXT_BALL_CLASSES` / `NEXT_BALL_MAP`: the 7-way   classification target for the (optional, disabled by default)   next-ball-outcome auxiliary head -- dot ball, 1/2/3/4/6 runs,   or a wicket.
- `LAMBDA_WIN_PROB/NEXT_BALL/SCORE`: the multi-task loss   weights -- `2.00` for win probability (the primary task,   weighted up), `0.0`/`0.0` for the two auxiliary heads   (disabled by default, per the module docstring above).

In [2]:
GAME_STATE_DIM = 24
PLAYER_EMBED_DIM = 32
INPUT_DIM = GAME_STATE_DIM + 3 * PLAYER_EMBED_DIM  # 24 + 96 = 120

TRANSFORMER_D_MODEL = 64
TRANSFORMER_NHEAD = 4
TRANSFORMER_NUM_LAYERS = 2
TRANSFORMER_DIM_FF = 128
TRANSFORMER_DROPOUT = 0.15
MAX_SEQ_LEN = 280

NEXT_BALL_CLASSES = 7  # [dot, 1, 2, 3, 4, 6, wicket]
NEXT_BALL_MAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 6: 5, "W": 6}

LAMBDA_WIN_PROB = 2.00
LAMBDA_NEXT_BALL = 0.0   # win-only default (see module docstring)
LAMBDA_SCORE = 0.0       # win-only default (see module docstring)

## `SinusoidalPositionalEncoding`

A Transformer's self-attention has no inherent sense of *order* -- without positional information, ball 1 and ball 100 look identical to the attention mechanism if their features happened to match. This class adds the classic sine/cosine positional encoding (from the original "Attention Is All You Need" paper) to each ball's projected feature vector, giving the model a way to distinguish *when* in the innings each ball occurred. The pattern is precomputed once for all `MAX_SEQ_LEN=280` positions and registered as a (non-trainable) buffer, so it doesn't need to be recomputed on every forward pass.

In [3]:
class SinusoidalPositionalEncoding(nn.Module):
    """Standard sinusoidal encoding over delivery index (0 ... MAX_SEQ_LEN-1)."""

    def __init__(self, d_model: int, max_len: int = MAX_SEQ_LEN, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, T, D)"""
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

## `_mlp()` -- shared output-head builder

All three prediction heads (win probability, next-ball outcome, score projection) share the exact same small architecture: Linear -> ReLU -> Dropout -> Linear. Defining this once as a factory function avoids repeating the same 4-line block three times with only the output dimension changing.

In [4]:
def _mlp(in_dim: int, hidden: int, out_dim: int, dropout: float = 0.1) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, out_dim),
    )

## `IPLTransformer` -- the main model

**Architecture, in order:**
1. `input_proj`: a Linear layer + LayerNorm projecting the    120-dim raw input (game state + 3 embeddings) into the    model's internal 64-dim working space.
2. `pos_enc`: adds the sinusoidal positional encoding described    above.
3. `transformer`: a standard PyTorch `TransformerEncoder` with    `norm_first=True` (Pre-LayerNorm, a well-known stability    improvement for training deep Transformers) and    `batch_first=True` (so tensors are shaped `(batch, seq,    features)` rather than PyTorch's older `(seq, batch,    features)` convention).
4. Three parallel output heads built from `_mlp()`: `win_head`    (scalar, sigmoid-activated -> a genuine probability),    `next_ball_head` (7-way logits), `score_head` (scalar,    softplus-activated so the projected score can never be    negative).

**`_causal_mask()`** is the critical piece that makes this a *causal* model: it builds an upper-triangular boolean mask so that when predicting ball `t`, the attention mechanism can only look at balls `0..t` -- never anything from the future. Without this mask the model could "cheat" by looking ahead to see how the innings actually ended.

**`forward()`** returns a dict with all three heads' outputs plus the raw encoder hidden states (`hidden`) -- the latter isn't used by anything in this project currently, but is kept available for anyone who wants to probe the model's internal representations directly.

**`_init_weights()`** applies Xavier-uniform initialisation to every Linear layer -- a standard, well-tested initialisation scheme that helps gradients flow reasonably at the start of training, rather than relying on PyTorch's (also reasonable, but less commonly recommended for Transformers) default init.

In [5]:
class IPLTransformer(nn.Module):
    """Multi-task causal Transformer for IPL innings modelling."""

    def __init__(
        self,
        input_dim: int = INPUT_DIM,
        d_model: int = TRANSFORMER_D_MODEL,
        nhead: int = TRANSFORMER_NHEAD,
        num_layers: int = TRANSFORMER_NUM_LAYERS,
        dim_ff: int = TRANSFORMER_DIM_FF,
        dropout: float = TRANSFORMER_DROPOUT,
        num_classes: int = NEXT_BALL_CLASSES,
    ):
        super().__init__()
        self.d_model = d_model

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
        )
        self.pos_enc = SinusoidalPositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )

        head_hidden = d_model // 2
        self.win_head = _mlp(d_model, head_hidden, 1, dropout)
        self.next_ball_head = _mlp(d_model, head_hidden, num_classes, dropout)
        self.score_head = _mlp(d_model, head_hidden, 1, dropout)

        self._init_weights()

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    @staticmethod
    def _causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
        """Upper-triangular mask (True = ignore) for causal attention."""
        return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()

    def forward(
        self,
        x: torch.Tensor,  # (B, T, INPUT_DIM)
        src_key_padding_mask: Optional[torch.Tensor] = None,  # (B, T) True=padding
    ) -> dict[str, torch.Tensor]:
        B, T, _ = x.shape

        h = self.input_proj(x)
        h = self.pos_enc(h)

        causal_mask = self._causal_mask(T, x.device)
        h = self.transformer(h, mask=causal_mask, src_key_padding_mask=src_key_padding_mask)

        win_prob = torch.sigmoid(self.win_head(h))
        next_ball = self.next_ball_head(h)
        score_proj = F.softplus(self.score_head(h))

        return {"win_prob": win_prob, "next_ball": next_ball, "score_proj": score_proj, "hidden": h}

## `MultiTaskLoss`

Weighted sum of three losses, one per output head:
- **Win probability**: binary cross-entropy (always active --   this is the primary task).
- **Next-ball outcome**: 7-way cross-entropy (only computed   when `lambda_next_ball > 0`).
- **Score projection**: Huber loss (robust to outliers, unlike   plain MSE -- appropriate for score prediction where the   occasional very-high-scoring innings shouldn't dominate the   gradient) with `delta=0.2` on the normalised `score/250`   scale (only computed when `lambda_score > 0`).

**The `if self.lnb > 0` / `if self.ls > 0` guards are the practical implementation of "win-only by default"**: with both lambdas at their default of 0.0, those two branches never execute, so no gradient at all flows through the next-ball or score heads during default training -- exactly replicating hrishav's own "Task 4" finding that disabling these auxiliary objectives improves the primary win-probability signal.

**`valid_mask`** ensures padded positions (added by `InningsDataset` below to make every innings the same length for batching) never contribute to any loss -- they're filtered out via boolean indexing (`.reshape(-1)[valid]`) before any loss function sees them.

In [6]:
class MultiTaskLoss(nn.Module):
    """Weighted combination of BCE(win) + CrossEntropy(next-ball) + Huber(score)."""

    def __init__(
        self,
        lambda_win: float = LAMBDA_WIN_PROB,
        lambda_next_ball: float = LAMBDA_NEXT_BALL,
        lambda_score: float = LAMBDA_SCORE,
    ):
        super().__init__()
        self.lw = lambda_win
        self.lnb = lambda_next_ball
        self.ls = lambda_score

    def forward(
        self,
        preds: dict[str, torch.Tensor],
        win_labels: torch.Tensor,     # (B, T) float 0/1
        nb_labels: torch.Tensor,      # (B, T) long 0-6
        score_labels: torch.Tensor,   # (B, T) float final_score / 250
        valid_mask: torch.Tensor,     # (B, T) bool True = real ball
    ) -> dict[str, torch.Tensor]:
        valid = valid_mask.reshape(-1)
        win_pred = preds["win_prob"].reshape(-1)[valid]
        win_tgt = win_labels.reshape(-1)[valid].float()

        loss_win = F.binary_cross_entropy(win_pred, win_tgt)
        total = self.lw * loss_win
        loss_nb = torch.tensor(0.0, device=win_pred.device)
        loss_sc = torch.tensor(0.0, device=win_pred.device)

        if self.lnb > 0:
            nb_pred = preds["next_ball"].reshape(-1, NEXT_BALL_CLASSES)[valid]
            nb_tgt = nb_labels.reshape(-1)[valid]
            loss_nb = F.cross_entropy(nb_pred, nb_tgt)
            total = total + self.lnb * loss_nb

        if self.ls > 0:
            sc_pred = preds["score_proj"].reshape(-1)[valid]
            sc_tgt = score_labels.reshape(-1)[valid].float()
            loss_sc = F.huber_loss(sc_pred, sc_tgt, delta=0.2)
            total = total + self.ls * loss_sc

        return {"loss": total, "loss_win": loss_win, "loss_nb": loss_nb, "loss_score": loss_sc}

## `InningsDataset` -- PyTorch Dataset wrapping one innings per sample

A standard PyTorch `Dataset` where each item is one full innings. Since innings have different lengths (a team bowled out in 15 overs vs. one that bats the full 20), every sample must be **padded to the same `max_len`** so a `DataLoader` can batch multiple innings together into one tensor.

For each item, `__getitem__`:
1. Truncates to at most `max_len` balls (in practice `max_len=280`    is generously larger than any real innings, so truncation    essentially never triggers).
2. Pads the feature matrix and next-ball labels with zeros out    to `max_len`.
3. Builds a `valid_mask` marking exactly which positions are    real balls vs. padding -- this is what `MultiTaskLoss` uses    to ignore padded positions.
4. Broadcasts the innings-level `win_label`/`score_label`    (constant for the whole innings) across every time step via    `torch.full`, since the loss function operates per-ball, not    per-innings.

In [7]:
class InningsDataset(torch.utils.data.Dataset):
    """One sample = one innings (sequence of deliveries)."""

    def __init__(self, innings_list: list[dict], max_len: int = MAX_SEQ_LEN):
        self.data = innings_list
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        item = self.data[idx]
        feats = item["features"]
        T = min(len(feats), self.max_len)
        feats = feats[:T]

        win_lbl = item["win_label"]
        nb_lbl = item["nb_labels"][:T]
        sc_lbl = item["score_label"]

        pad = self.max_len - T
        feat_pad = torch.zeros(pad, feats.shape[1], dtype=torch.float32)
        nb_pad = torch.zeros(pad, dtype=torch.long)
        valid = torch.zeros(self.max_len, dtype=torch.bool)
        valid[:T] = True

        features_t = torch.cat([torch.tensor(feats, dtype=torch.float32), feat_pad], dim=0)
        nb_t = torch.cat([torch.tensor(nb_lbl, dtype=torch.long), nb_pad], dim=0)

        win_t = torch.full((self.max_len,), float(win_lbl))
        score_t = torch.full((self.max_len,), float(sc_lbl))

        return {
            "features": features_t,
            "win_labels": win_t,
            "nb_labels": nb_t,
            "score_labels": score_t,
            "valid_mask": valid,
        }

## `PlayerEmbedTable` -- player identity embeddings

**Ported from `project_hrishav/player_graph.py`.** A plain `nn.Embedding` lookup: every player gets a learnable 32-dim vector representing "who they are" as far as the model is concerned. `padding_idx=0` reserves index 0 as a permanently zero embedding (used for padding/unknown players), and Xavier initialisation is applied to every *real* player row (`weight.data[1:]`, skipping the padding row).

**Why there's no GraphSAGE GNN here** (hrishav's original project builds player embeddings from a batter/bowler matchup graph via a pretrained GNN, as an alternative to this simple table): hrishav's own 3-seed ablation study found **no statistically significant advantage** from the GNN embeddings over this much simpler random-init table, at their data scale. Since the GNN also requires the optional, sometimes-tricky-to-install `torch_geometric` package, and the evidence says it isn't worth it, this port uses only `PlayerEmbedTable` -- a decision backed by hrishav's own data, not a corner cut during the port.

In [8]:
class PlayerEmbedTable(nn.Module):
    """
    Simple nn.Embedding lookup for player identity, fine-tuned jointly with
    the Transformer. Ported from project_hrishav/player_graph.py.

    hrishav's own 3-seed ablation (DEEP_COMPARISON.md Table 4/5) found no
    statistically significant advantage from GraphSAGE GNN embeddings over
    this random-init table at their data scale, so the GNN (which requires
    the optional torch_geometric dependency) was not ported -- this is the
    only embedding source used here, matching hrishav's own evidence-based
    conclusion rather than cutting a feature that mattered.
    """

    def __init__(self, num_players: int, embed_dim: int = PLAYER_EMBED_DIM):
        super().__init__()
        self.embed = nn.Embedding(num_players + 1, embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.embed.weight.data[1:])

    def forward(self, player_ids: torch.Tensor) -> torch.Tensor:
        return self.embed(player_ids)